In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

In [ ]:
# Display settings
pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8,5)

In [ ]:
df = pd.read_csv(r"C:\Users\dadof\Downloads\cybersecurity_intrusion_data (1).csv")
df

In [ ]:
# Define target BEFORE dropping anything
y = df["attack_detected"]

# Drop ID + target from features
X = df.drop(["attack_detected", "session_id"], axis=1)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

In [ ]:
print(df.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
num_cols = df.select_dtypes(include=["int64","float64"]).columns.tolist()
num_cols

In [ ]:
num_cols = [
    'network_packet_size',
    'login_attempts',
    'session_duration',
    'ip_reputation_score',
    'failed_logins',
    'unusual_time_access'
]

In [ ]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols

In [ ]:
cat_cols = [
    'protocol_type',
    'encryption_used',
    'browser_type'
]

In [ ]:
print(df.columns.tolist())

### LOGISTIC REGRESSION MODEL 

In [ ]:
# Preprocessing Pipeline 
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

categorical_pipeline

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])
preprocessor

In [ ]:
log_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])
log_pipeline

In [ ]:
print(df.columns.tolist())

In [ ]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

In [ ]:
log_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = log_pipeline.predict(X_test)
y_pred
y_proba = log_pipeline.predict_proba(X_test)[:, 1]
y_proba

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### RANDOMFOREST MODEL 

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

rf_pipeline

In [ ]:
rf_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)
y_pred_rf
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]
y_proba_rf

In [ ]:
print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf))

### DECISIONTREE 

In [ ]:
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,        # control overfitting
        random_state=42
    ))
])
dt_pipeline

In [ ]:
dt_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_dt = dt_pipeline.predict(X_test)
y_pred_dt
y_proba_dt = dt_pipeline.predict_proba(X_test)[:, 1]
y_proba_dt

In [ ]:
print("Decision Tree Results:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Get trained tree
tree_model = dt_pipeline.named_steps["classifier"]

plt.figure(figsize=(20,10))
plot_tree(
    tree_model,
    filled=True,
    feature_names=dt_pipeline.named_steps["preprocessor"].get_feature_names_out(),
    class_names=["No Attack", "Attack"],
    max_depth=3
)
plt.show()

In [ ]:
# Feature Importance 
feature_names = dt_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = dt_pipeline.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

importance_df.head(10)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "Logistic Regression": log_pipeline,
    "Decision Tree": dt_pipeline,
    "Random Forest": rf_pipeline
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
    })

comparison_df = pd.DataFrame(results)
comparison_df

### Hyper ParameterTuning 

In [ ]:
from sklearn.model_selection import GridSearchCV

dt_param_grid = {
    "classifier__max_depth": [3, 5, 10, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 5]
}
dt_param_grid

dt_grid = GridSearchCV(
    dt_pipeline,
    dt_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)
dt_grid

dt_grid.fit(X_train, y_train)

print("Best Decision Tree Parameters:")
print(dt_grid.best_params_)

In [ ]:
best_dt = dt_grid.best_estimator_
best_dt

y_pred_dt = best_dt.predict(X_test)
y_pred_dt
y_proba_dt = best_dt.predict_proba(X_test)[:, 1]
y_proba_dt

print("Tuned Decision Tree Results:")

In [ ]:
rf_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}
rf_param_grid

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)
rf_param_grid

rf_grid.fit(X_train, y_train)

print("Best Random Forest Parameters:")
print(rf_grid.best_params_)

In [ ]:
best_rf = rf_grid.best_estimator_
best_rf
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]
y_proba_rf

In [ ]:
print(X.columns)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Numeric pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

# Full model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [ ]:
model.fit(X_train,y_train)

In [ ]:
import joblib
joblib.dump(rf_pipeline, "best_model.pkl")